# Incident Triage Agent — a LangChain + LangGraph primer

A self-contained tour of **LangChain 1.x** and **LangGraph 1.x** for someone who
has built agents before but never used these libraries. Nothing here imports the
`florence` package — it stands alone.

The running example is a tiny **clinical incident-triage assistant**: a nurse/manager
describes an incident, the agent can call tools (on-call lookup, severity
guidelines, ticket creation) and decides what to do.

### The mental model (how these two libraries divide the work)

| Layer | What it owns | Key objects |
|---|---|---|
| **LangChain** | The *primitives*: chat models, the message types, tool definitions, structured output | `init_chat_model`, `HumanMessage`/`AIMessage`/`ToolMessage`, `@tool` |
| **LangGraph** | The *runtime*: a state machine (nodes + edges) that drives the loop, plus persistence/streaming | `StateGraph`, `MessagesState`, `ToolNode`, checkpointers |

We'll go bottom-up:

1. Talk to a model (messages in, message out)
2. Define tools and run the tool-call loop **by hand** (so nothing is magic)
3. Collapse that loop into one line with LangChain's prebuilt `create_agent`
4. Rebuild the same loop **from scratch** in LangGraph to see what `create_agent` generates
5. Add memory (checkpointer + `thread_id`) for multi-turn
6. Force **structured output** with a Pydantic schema
7. Stream the agent's steps

## 0. Setup

Only an OpenAI API key is needed. Everything else is already in this project's
virtualenv (`langchain`, `langgraph`, `langchain-openai`).

In [ ]:
import getpass
import os

# Pull the key from the environment (or .env if you exported it); prompt if absent.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

# Any current model works because we use the Responses API below. The GPT-5
# family are reasoning models; swap in whatever your account has.
MODEL = "gpt-5.6-luna"

## 1. The model + the message types

`init_chat_model` is LangChain 1.x's provider-agnostic factory; the string form
lets you swap providers (`"anthropic:claude-..."`, `"groq:..."`) without changing
code.

**Up-to-date practice — use the Responses API.** OpenAI's current models (the
GPT-5 family and other reasoning models) expect the `/v1/responses` endpoint; the
older `/v1/chat/completions` endpoint **rejects function-calling** on them (that's
the `400 ... use /v1/responses` error). In `langchain-openai` you opt in with
`use_responses_api=True`. It also works for older models, so it's the safe,
forward-compatible default.

Two more reasoning-model gotchas folded into the cell below:

- **Don't pin `temperature`.** Reasoning models accept only the default — passing
  `temperature=0` is a 400. Omit it; steer effort with `reasoning_effort`
  (`"minimal"` → `"high"`) instead.
- **Read `.text`, not `.content`.** The Responses API can return a *list* of
  content blocks (reasoning, text, tool calls); the `.text` property always gives
  you the plain string.

A chat model takes a **list of messages** and returns a single **`AIMessage`**.
The message classes are the vocabulary you'll see everywhere:

- `SystemMessage` — instructions / persona
- `HumanMessage` — user input
- `AIMessage` — model output (text **and/or** `tool_calls`)
- `ToolMessage` — the result of running a tool, fed back to the model

A bare string is accepted as shorthand for a single `HumanMessage`.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

# use_responses_api=True -> /v1/responses (required for GPT-5 / reasoning models
# when tools are involved; harmless for older models). No temperature: reasoning
# models reject a custom value. reasoning_effort is the knob you tune instead.
model = init_chat_model(MODEL, use_responses_api=True, reasoning_effort="low")

resp = model.invoke(
    [
        SystemMessage("You are a terse clinical incident-triage assistant."),
        HumanMessage("A patient fall was reported on Ward 4B. What are the first steps?"),
    ]
)

print(type(resp).__name__)      # -> AIMessage
print(resp.text)                # .text = plain string; .content may be blocks
# `.invoke` is sync; `.ainvoke` is the async twin. `.stream`/`.astream` yield chunks.

## 2. Tools, and the tool-call loop by hand

The `@tool` decorator turns a plain typed function into a `BaseTool`. The
docstring and type hints become the schema the model sees — so write them for the
model, not just for humans.

`model.bind_tools([...])` returns a new model that *may* respond with
`tool_calls` instead of final text. The model never runs your code; it only asks
for a call. **You** execute it and feed a `ToolMessage` back. That request →
execute → feed-back cycle is the whole "agent loop" — let's run it manually once.

In [ ]:
from langchain_core.tools import tool

# Fake backends so the notebook is self-contained — no real DB/network.
_ONCALL = {"cardiology": "Dr. Patel", "ward 4b": "Nurse Okafor", "er": "Dr. Lin"}
_TICKETS: list[dict] = []


@tool
def lookup_oncall(department: str) -> str:
    """Return the name of the clinician on call for a department."""
    return _ONCALL.get(department.lower(), "No on-call clinician found; page the duty manager.")


@tool
def severity_guidelines(category: str) -> str:
    """Return triage severity guidance for an incident category (e.g. 'fall', 'medication')."""
    table = {
        "fall": "Falls: assess for injury. Head strike or anticoagulated patient => SEV-1.",
        "medication": "Med errors: wrong dose/route reaching the patient => SEV-1, else SEV-2.",
    }
    return table.get(category.lower(), "Unknown category; default to SEV-2 and escalate.")


@tool
def create_incident_ticket(ward: str, category: str, severity: str) -> str:
    """Create an incident ticket and return its id."""
    ticket = {"id": f"INC-{1000 + len(_TICKETS)}", "ward": ward, "category": category, "severity": severity}
    _TICKETS.append(ticket)
    return ticket["id"]


TOOLS = [lookup_oncall, severity_guidelines, create_incident_ticket]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

model_with_tools = model.bind_tools(TOOLS)

In [ ]:
# One turn of the loop, fully manual.
messages = [
    SystemMessage("You are a clinical incident-triage assistant. Use tools when helpful."),
    HumanMessage("Who is on call for Ward 4B?"),
]

ai: AIMessage = model_with_tools.invoke(messages)
messages.append(ai)

# Instead of final text, the model asked for tool calls:
print("tool_calls:", ai.tool_calls)

# Execute each requested call and append its result as a ToolMessage.
for call in ai.tool_calls:
    result = TOOLS_BY_NAME[call["name"]].invoke(call["args"])
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Send everything back so the model can turn the tool result into an answer.
final = model_with_tools.invoke(messages)
print("final:", final.text)

That by-hand loop — *call model → if tool_calls, run them → append results →
call model again → repeat until no tool_calls* — is exactly what every "agent"
abstraction below automates. Keep it in mind; the rest is ergonomics.

## 3. The one-liner: LangChain's `create_agent`

LangChain 1.x ships a prebuilt ReAct agent. `create_agent(model, tools,
system_prompt=...)` builds the whole loop from section 2 and returns a **compiled
LangGraph graph**. You invoke it with `{"messages": [...]}` and get back the full
message list (your input + every tool call + the final answer).

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model,  # or just the string MODEL
    tools=TOOLS,
    system_prompt="You are a clinical incident-triage assistant. Use tools when helpful.",
)

result = agent.invoke(
    {"messages": [HumanMessage("A fall on Ward 4B, patient is on blood thinners. "
                               "File a ticket at the right severity and tell me who's on call.")]}
)

# Walk the transcript the agent produced — note the tool calls and results inline.
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# The tool actually ran — our fake store now has the ticket:
_TICKETS

## 4. The same agent, built from scratch in LangGraph

`create_agent` is convenient but opaque. Here's the identical behaviour assembled
by hand so you can see the moving parts — this is the core LangGraph API.

**State.** A graph threads a shared state dict through its nodes. `MessagesState`
is a prebuilt schema with one key, `messages`, whose **reducer** (`add_messages`)
*appends* rather than overwrites. So each node returns `{"messages": [...]}` and
the new messages are merged in.

**Nodes** are functions `state -> partial_state_update`.

**Edges** wire nodes together. A **conditional edge** picks the next node from the
current state — here, "did the last AIMessage ask for tools?". `tools_condition`
is a prebuilt that returns `"tools"` if so, else `END`.

**`ToolNode`** is a prebuilt node that reads the last AIMessage's `tool_calls`,
runs them, and appends the `ToolMessage`s — the loop body from section 2.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

SYSTEM = SystemMessage("You are a clinical incident-triage assistant. Use tools when helpful.")


def call_model(state: MessagesState) -> dict:
    """The 'reasoning' node: prepend the system prompt and ask the model."""
    response = model_with_tools.invoke([SYSTEM, *state["messages"]])
    return {"messages": [response]}  # appended via the add_messages reducer


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(TOOLS))

builder.add_edge(START, "model")
# From the model node, branch: if it asked for tools -> "tools", else -> END.
builder.add_conditional_edges("model", tools_condition, {"tools": "tools", END: END})
builder.add_edge("tools", "model")  # after running tools, loop back to the model

graph = builder.compile()

In [ ]:
# Visualize the state machine (START -> model <-> tools -> END).
try:
    from IPython.display import Image, display

    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Rendering needs network/graphviz; the ASCII/mermaid text always works:
    print(graph.get_graph().draw_mermaid())

In [ ]:
out = graph.invoke(
    {"messages": [HumanMessage("Medication error on the ER floor reached the patient. "
                               "What severity, and who do I call?")]}
)
for m in out["messages"]:
    m.pretty_print()

## 5. Memory: multi-turn with a checkpointer

So far each `invoke` starts cold. To hold a conversation, compile the graph with a
**checkpointer**. It snapshots state after every step, keyed by a `thread_id` you
pass in `config`. Re-invoking with the same `thread_id` resumes that thread, so
you only send the *new* message — history is reloaded automatically.

`InMemorySaver` is the dev checkpointer; swap in the Postgres/SQLite savers for
production persistence without touching graph code.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_graph = builder.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "ward-4b-shift"}}

# Turn 1
r1 = memory_graph.invoke({"messages": [HumanMessage("Who's on call for the ER?")]}, cfg)
print("A:", r1["messages"][-1].text)

# Turn 2 — note we never repeat "ER"; the checkpointer still has the context.
r2 = memory_graph.invoke({"messages": [HumanMessage("And what's their severity rule for med errors?")]}, cfg)
print("A:", r2["messages"][-1].text)

# A different thread_id is a clean slate:
r3 = memory_graph.invoke({"messages": [HumanMessage("What did I just ask you?")]},
                         {"configurable": {"thread_id": "someone-else"}})
print("other thread:", r3["messages"][-1].text)

## 6. Structured output

Agents often need to end on a typed object, not prose. Pass a Pydantic model as
`response_format` to `create_agent`; the final result carries a validated
`structured_response`. (Under the hood this is just one more tool/parse step the
graph runs at the end.)

In [ ]:
from pydantic import BaseModel, Field


class TriageResult(BaseModel):
    """Structured outcome of triaging one incident."""

    category: str = Field(description="Incident category, e.g. 'fall' or 'medication'.")
    severity: str = Field(description="One of SEV-1, SEV-2, SEV-3.")
    on_call: str = Field(description="Name of the clinician to contact.")
    rationale: str = Field(description="One sentence justifying the severity.")


structured_agent = create_agent(
    model,
    tools=TOOLS,
    system_prompt="You are a clinical incident-triage assistant. Use tools, then summarize.",
    response_format=TriageResult,
)

res = structured_agent.invoke(
    {"messages": [HumanMessage("Patient fall on Ward 4B, head strike, on anticoagulants.")]}
)
res["structured_response"]  # a validated TriageResult instance

## 7. Streaming the agent's steps

`.invoke` waits for the whole run. `.stream` emits incrementally. Two useful modes:

- `stream_mode="updates"` — one event per node as it finishes (great for logs/UX:
  "model thinking", "ran tool X").
- `stream_mode="messages"` — token-by-token streaming of the model's text.

Both return generators; `.astream` is the async version for your FastAPI SSE/WebSocket layer.

In [ ]:
print("--- updates mode: one event per node ---")
for event in graph.stream(
    {"messages": [HumanMessage("File a SEV-2 ticket for a minor fall on Ward 4B.")]},
    stream_mode="updates",
):
    for node, update in event.items():
        last = update["messages"][-1]
        kind = "tool_calls" if getattr(last, "tool_calls", None) else "text"
        print(f"[{node}] -> {kind}: {last.text or last.tool_calls}")

In [ ]:
print("--- messages mode: token streaming of the final answer ---")
for token, meta in graph.stream(
    {"messages": [HumanMessage("In one sentence, why is a head-strike fall SEV-1?")]},
    stream_mode="messages",
):
    if token.text:
        print(token.text, end="", flush=True)
print()

## Recap — the map you now have

- **LangChain** gives you the primitives: `init_chat_model`, the message classes,
  `@tool`, `bind_tools`, structured output.
- **The agent loop** is just: call model → run requested tools → feed results back
  → repeat until no tool calls (section 2). Everything else automates this.
- **`create_agent`** is the batteries-included version (one line).
- **LangGraph** is the runtime underneath: `StateGraph` + `MessagesState` (the
  `add_messages` reducer appends), nodes/edges, `tools_condition` + `ToolNode`
  for the branch-and-loop, `compile()` to get a runnable.
- **Checkpointer + `thread_id`** = memory; swap the saver for real persistence.
- **`.stream(...)`** with `updates`/`messages` modes feeds a real-time transport.

Next steps to explore: custom nodes between model and tools (guardrails, PHI
redaction), `interrupt_before=["tools"]` for human-in-the-loop approval, and
subgraphs for multi-agent setups.